# Debt waterfall & cash sweep

**Priority of payments**: cash pays senior cash interest first, then mandatory senior principal, then subordinated interest, then residual to equity. Stress with **lower UFCF**.

## Concept

`WaterfallSpec` defines the payment order and available-cash node. `ModelBuilder.waterfall` attaches that policy to a capital structure, and the Rust evaluator executes it against contractual debt cashflows. Live facilities still require deal-specific day counts, PIK toggles, and restricted-payment terms.

In [1]:
import json
from datetime import date

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import MarketContext
from finstack_quant.core.money import Money
from finstack_quant.statements import Evaluator, ModelBuilder, WaterfallSpec

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]


def build_waterfall_model(model_id: str, available_cash: list[float]):
    builder = ModelBuilder(model_id)
    builder.periods("2025Q1..Q4", None)
    builder.value("cash_flow_available", list(zip(PERIODS, available_cash)))
    builder.add_bond(
        id="SENIOR",
        notional=Money(100.0, Currency("USD")),
        coupon_rate=0.06,
        issue_date=date(2025, 1, 1),
        maturity_date=date(2025, 12, 31),
        discount_curve_id="USD-OIS",
    )
    waterfall = WaterfallSpec(
        priority_of_payments=["fees", "interest", "amortization", "equity"],
        available_cash_node="cash_flow_available",
    )
    waterfall.validate()
    builder.waterfall(waterfall)
    return builder.build(), waterfall


def evaluate_waterfall(model):
    result = Evaluator().evaluate_with_market(model, MarketContext(), date(2025, 1, 1))
    return result, json.loads(result.to_json())["cs_cashflows"]


spec, waterfall = build_waterfall_model("waterfall-base", [20.0, 18.0, 16.0, 14.0])
base_result, base_cashflows = evaluate_waterfall(spec)
print("Declarative priority:", waterfall.priority_of_payments)
for period in PERIODS:
    totals = base_cashflows["totals"][period]
    equity = base_cashflows["equity_distribution"][period]
    print(
        period,
        "interest=",
        float(totals["interest_expense_cash"]["amount"]),
        "principal=",
        float(totals["principal_payment"]["amount"]),
        "equity=",
        float(equity["amount"]),
    )

Declarative priority: ['fees', 'interest', 'amortization', 'equity']
2025Q1 interest= 0.0 principal= 0.0 equity= 20.0
2025Q2 interest= 3.281666666666665 principal= 0.0 equity= 14.71833333333334
2025Q3 interest= 0.0 principal= 0.0 equity= 16.0
2025Q4 interest= 3.3 principal= 10.7 equity= 0.0


In [2]:
print("Waterfall stress through the same declarative engine")
stress_spec, _ = build_waterfall_model("waterfall-stress", [9.0, 9.0, 9.0, 9.0])
stress_result, stress_cashflows = evaluate_waterfall(stress_spec)

for period in PERIODS:
    totals = stress_cashflows["totals"][period]
    equity = stress_cashflows["equity_distribution"][period]
    print(
        period,
        "interest=",
        float(totals["interest_expense_cash"]["amount"]),
        "principal=",
        float(totals["principal_payment"]["amount"]),
        "equity=",
        float(equity["amount"]),
    )

print("Both base and stress use WaterfallSpec via ModelBuilder.waterfall; no parallel DSL cascade.")

Waterfall stress through the same declarative engine
2025Q1 interest= 0.0 principal= 0.0 equity= 9.0
2025Q2 interest= 3.281666666666665 principal= 0.0 equity= 5.71833333333334
2025Q3 interest= 0.0 principal= 0.0 equity= 9.0
2025Q4 interest= 3.3 principal= 5.7 equity= 0.0
Both base and stress use WaterfallSpec via ModelBuilder.waterfall; no parallel DSL cascade.


## Declarative specification

The examples above use `WaterfallSpec` and `ModelBuilder.waterfall` as the only execution path. Optional `EcfSweepSpec` and `PikToggleSpec` extend the same Rust-owned waterfall without creating a parallel set of hand-coded formulas.


In [3]:
print("WaterfallSpec:", waterfall)
waterfall.validate()
print("Attached model contains waterfall:", '"waterfall"' in spec.to_json())

WaterfallSpec: WaterfallSpec(priority=["fees", "interest", "amortization", "equity"], ecf_sweep=false, pik_toggle=false)
Attached model contains waterfall: True


## Takeaways

- `WaterfallSpec` makes legal payment priority explicit.
- `ModelBuilder.waterfall` routes both base and stress cases through the Rust capital-structure engine.
- Available cash caps interest, principal, and residual equity without a parallel hand-coded cascade.